# LeetCode 1480 — Running Sum of 1d Array bằng Python

**Link đề:** https://leetcode.com/problems/running-sum-of-1d-array/
**Độ khó:** Easy · **Chủ đề:** Array, Prefix Sum (tổng tiền tố)

```python
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Hiểu **running sum** (tổng chạy / tổng cộng dồn) và **prefix sum** (tổng tiền tố) là gì qua ví dụ tay.
2. Viết lời giải trực giác bằng `sum(nums[:i+1])` và thấy vì sao nó `O(n²)`.
3. Rút ra công thức then chốt `running[i] = running[i-1] + nums[i]` → lời giải `O(n)`.
4. Biết 4 cách cài đặt: list mới, sửa tại chỗ (in-place), `itertools.accumulate`, và dạng prefix có số 0 ở đầu.
5. Dùng prefix sum để trả lời nhanh "tổng đoạn `[l, r]`" — nền tảng cho rất nhiều bài khác.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tổng cộng dồn

Cho mảng `nums`. Định nghĩa **running sum** của mảng là:

```
runningSum[i] = nums[0] + nums[1] + ... + nums[i]
```

Trả về mảng running sum.

| `nums` | Output | Giải thích |
|---|---|---|
| `[1,2,3,4]` | `[1,3,6,10]` | `[1, 1+2, 1+2+3, 1+2+3+4]` |
| `[1,1,1,1,1]` | `[1,2,3,4,5]` | mỗi bước cộng thêm 1 |
| `[3,1,2,10,1]` | `[3,4,6,16,17]` | |

**Constraints:** `1 <= nums.length <= 1000`, `-10^6 <= nums[i] <= 10^6`.

Chú ý: số có thể **âm** → running sum **không nhất thiết tăng**. Kết quả có cùng độ dài với `nums`.

Hình dung đời thường: `nums` là số tiền bạn nạp/rút mỗi ngày; running sum là **số dư tài khoản** cuối mỗi ngày.

In [1]:
# Ví dụ đời thường: nạp/rút tiền mỗi ngày (âm = rút)
giao_dich = [100, -30, 50, -20]
so_du = 0
for ngay, tien in enumerate(giao_dich):
    so_du = so_du + tien
    print(f"Ngày {ngay}: giao dịch {tien:>4} -> số dư {so_du}")

Ngày 0: giao dịch  100 -> số dư 100
Ngày 1: giao dịch  -30 -> số dư 70
Ngày 2: giao dịch   50 -> số dư 120
Ngày 3: giao dịch  -20 -> số dư 100


Bạn vừa tự viết running sum mà không biết! Ý tưởng chính của bài chỉ gói gọn trong dòng `so_du = so_du + tien`. Phần còn lại của notebook giải thích kỹ vì sao nó đúng, các cách cài đặt và ứng dụng.

## 2. Nền tảng Python cần biết

### 2.1. Chỉ số (index) và slicing `nums[:k]`

- `nums[i]` là phần tử thứ `i` (đếm từ 0).
- `nums[:k]` là list con gồm `k` phần tử đầu: chỉ số `0, 1, ..., k-1` (không lấy `k`).
- Vậy `nums[:i+1]` là các phần tử từ `0` tới `i` — đúng đoạn cần cộng cho `runningSum[i]`.

In [2]:
nums = [3, 1, 2, 10, 1]
for i in range(len(nums)):
    print(f"i={i}: nums[:i+1] = {nums[:i+1]}")

i=0: nums[:i+1] = [3]
i=1: nums[:i+1] = [3, 1]
i=2: nums[:i+1] = [3, 1, 2]
i=3: nums[:i+1] = [3, 1, 2, 10]
i=4: nums[:i+1] = [3, 1, 2, 10, 1]


### 2.2. `sum()` và `append()`

- `sum(list)` cộng tất cả phần tử, `sum([])` bằng `0`.
- `result.append(x)` thêm `x` vào **cuối** list `result`.

In [3]:
print(sum([3, 1, 2]))     # 6
print(sum([]))            # 0
result = []
result.append(3)
result.append(4)
print(result)             # [3, 4]

6
0
[3, 4]


### 2.3. Toán tử gán cộng `+=`

`x += y` là cách viết gọn của `x = x + y`. Với list, `nums[i] += nums[i-1]` nghĩa là `nums[i] = nums[i] + nums[i-1]`.

In [4]:
total = 10
total += 5
print(total)        # 15
a = [1, 2, 3]
a[1] += a[0]
print(a)            # [1, 3, 3]

15
[1, 3, 3]


## 3. Cách 1 — Trực giác: cộng lại từ đầu cho mỗi vị trí

Dịch thẳng định nghĩa: với mỗi `i`, tính `sum(nums[:i+1])`.

In [5]:
def running_sum_bruteforce(nums: list[int]) -> list[int]:
    result = []
    for i in range(len(nums)):
        result.append(sum(nums[:i + 1]))   # cộng lại toàn bộ đoạn 0..i
    return result

print(running_sum_bruteforce([1, 2, 3, 4]))       # [1, 3, 6, 10]
print(running_sum_bruteforce([1, 1, 1, 1, 1]))    # [1, 2, 3, 4, 5]
print(running_sum_bruteforce([3, 1, 2, 10, 1]))   # [3, 4, 6, 16, 17]

[1, 3, 6, 10]
[1, 2, 3, 4, 5]
[3, 4, 6, 16, 17]


### Code nộp LeetCode — Cách 1

Với `n <= 1000` vẫn được chấp nhận, nhưng chậm hơn cần thiết.

In [6]:
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        return [sum(nums[:i + 1]) for i in range(len(nums))]

print(Solution().runningSum([1, 2, 3, 4]))

[1, 3, 6, 10]


Dòng trên dùng **list comprehension** `[biểu_thức for i in ...]` — cách viết gọn của vòng `for` + `append`. Hai đoạn sau cho cùng kết quả:

In [7]:
squares_loop = []
for i in range(4):
    squares_loop.append(i * i)
squares_comp = [i * i for i in range(4)]
print(squares_loop, squares_comp, squares_loop == squares_comp)

[0, 1, 4, 9] [0, 1, 4, 9] True


## 4. Nút thắt — ta cộng đi cộng lại cùng một thứ

Với `i = 3` ta cộng `1+2+3+4`. Nhưng ở `i = 2` ta **vừa** cộng xong `1+2+3 = 6`! Cộng lại từ đầu là lãng phí.

Đếm số phép cộng của cách 1: vị trí `i` tốn `i+1` phép → tổng `1 + 2 + ... + n = n(n+1)/2` → `O(n²)`. Slicing `nums[:i+1]` còn tạo list mới dài `i+1` mỗi lần → tốn thêm thời gian và bộ nhớ tạm.

In [8]:
def count_additions_bruteforce(n):
    return sum(i + 1 for i in range(n))

for n in [4, 100, 1000]:
    print(f"n={n:>4}: cách 1 cộng {count_additions_bruteforce(n):>7} số | cách tối ưu chỉ cần {n} phép cộng")

n=   4: cách 1 cộng      10 số | cách tối ưu chỉ cần 4 phép cộng
n= 100: cách 1 cộng    5050 số | cách tối ưu chỉ cần 100 phép cộng
n=1000: cách 1 cộng  500500 số | cách tối ưu chỉ cần 1000 phép cộng


## 5. Insight — mỗi kết quả = kết quả trước + phần tử hiện tại

Gợi ý của đề: *tính phần tử thứ `i` từ phần tử thứ `i-1`*.

```
running[i]   = nums[0] + ... + nums[i-1] + nums[i]
             = (nums[0] + ... + nums[i-1]) + nums[i]
             = running[i-1]                + nums[i]
```

Với `running[0] = nums[0]`.

Ví dụ `[3, 1, 2, 10, 1]`:

```
running[0] = 3
running[1] = 3  + 1  = 4
running[2] = 4  + 2  = 6
running[3] = 6  + 10 = 16
running[4] = 16 + 1  = 17
```

In [9]:
nums = [3, 1, 2, 10, 1]
r0 = nums[0]
r1 = r0 + nums[1]
r2 = r1 + nums[2]
print(r0, r1, r2)    # 3 4 6

3 4 6


## 6. Cách 2 — Biến `total` chạy + list mới (lời giải chính)

Mổ xẻ:
- **Đoạn A:** `total = 0` — tổng các phần tử đã đi qua (ban đầu chưa đi qua gì).
- **Đoạn B:** với mỗi `x` trong `nums`: `total += x` → giờ `total` là tổng từ đầu tới `x`.
- **Đoạn C:** `result.append(total)` — ghi lại số dư sau bước này.

Bắt đầu từ `total = 0` giúp **không cần xử lý riêng** phần tử đầu: `0 + nums[0] = nums[0]`.

In [10]:
def running_sum_total(nums: list[int]) -> list[int]:
    result = []
    total = 0                 # tổng các phần tử đã duyệt
    for x in nums:
        total += x            # cộng thêm phần tử hiện tại
        result.append(total)  # lưu tổng cộng dồn tới đây
    return result

print(running_sum_total([1, 2, 3, 4]))
print(running_sum_total([1, 1, 1, 1, 1]))
print(running_sum_total([3, 1, 2, 10, 1]))

[1, 3, 6, 10]
[1, 2, 3, 4, 5]
[3, 4, 6, 16, 17]


## 7. TRACE từng bước

Ví dụ `nums = [3, 1, 2, 10, 1]`:

| i | x = nums[i] | total trước | total sau = trước + x | result sau |
|---:|---:|---:|---:|---|
| 0 | 3 | 0 | 3 | `[3]` |
| 1 | 1 | 3 | 4 | `[3, 4]` |
| 2 | 2 | 4 | 6 | `[3, 4, 6]` |
| 3 | 10 | 6 | 16 | `[3, 4, 6, 16]` |
| 4 | 1 | 16 | 17 | `[3, 4, 6, 16, 17]` |

In [11]:
def trace_running_sum(nums):
    result, total = [], 0
    for i, x in enumerate(nums):
        before = total
        total += x
        result.append(total)
        print(f"i={i}, x={x:>3}: total {before:>3} + {x:>3} = {total:>3} -> result = {result}")
    return result

trace_running_sum([3, 1, 2, 10, 1])

i=0, x=  3: total   0 +   3 =   3 -> result = [3]
i=1, x=  1: total   3 +   1 =   4 -> result = [3, 4]
i=2, x=  2: total   4 +   2 =   6 -> result = [3, 4, 6]
i=3, x= 10: total   6 +  10 =  16 -> result = [3, 4, 6, 16]
i=4, x=  1: total  16 +   1 =  17 -> result = [3, 4, 6, 16, 17]


[3, 4, 6, 16, 17]

Trace với số **âm** để thấy running sum có thể giảm:

In [12]:
trace_running_sum([5, -2, -4, 3])

i=0, x=  5: total   0 +   5 =   5 -> result = [5]
i=1, x= -2: total   5 +  -2 =   3 -> result = [5, 3]
i=2, x= -4: total   3 +  -4 =  -1 -> result = [5, 3, -1]
i=3, x=  3: total  -1 +   3 =   2 -> result = [5, 3, -1, 2]


[5, 3, -1, 2]

## 8. Cách 3 — Sửa tại chỗ (in-place), bộ nhớ phụ `O(1)`

Vì `running[i] = running[i-1] + nums[i]`, ta có thể ghi đè ngay vào `nums`: sau khi xử lý xong vị trí `i-1`, `nums[i-1]` đã là running sum, nên chỉ cần `nums[i] += nums[i-1]`.

Chú ý: phải bắt đầu từ `i = 1` (vị trí 0 giữ nguyên) và đi **từ trái sang phải**.

In [13]:
def running_sum_inplace(nums: list[int]) -> list[int]:
    for i in range(1, len(nums)):
        nums[i] += nums[i - 1]   # nums[i-1] lúc này ĐÃ là tổng cộng dồn
    return nums

data = [3, 1, 2, 10, 1]
print(running_sum_inplace(data))
print("data sau khi gọi:", data, "<- input đã bị thay đổi")

[3, 4, 6, 16, 17]
data sau khi gọi: [3, 4, 6, 16, 17] <- input đã bị thay đổi


**Đánh đổi:** tiết kiệm bộ nhớ nhưng **phá input**. Trên LeetCode chấp nhận được; trong code thật hãy cân nhắc (người gọi có thể vẫn cần mảng gốc). Nếu cần giữ input, gọi với bản sao: `running_sum_inplace(nums[:])`.

### Vì sao đi từ phải sang trái lại sai?

In [14]:
def running_sum_inplace_wrong_direction(nums):
    for i in range(len(nums) - 1, 0, -1):   # SAI: nums[i-1] chưa được cập nhật
        nums[i] += nums[i - 1]
    return nums

print(running_sum_inplace_wrong_direction([1, 2, 3, 4]), "| đúng phải là", running_sum_total([1, 2, 3, 4]))

[1, 3, 5, 7] | đúng phải là [1, 3, 6, 10]


Đi ngược, `nums[i-1]` vẫn là giá trị **gốc**, nên mỗi ô chỉ là tổng hai phần tử kề nhau.

## 9. Cách 4 — `itertools.accumulate` (thư viện chuẩn)

Module `itertools` có hàm `accumulate(iterable)` sinh ra đúng các tổng cộng dồn. Nó trả về một **iterator** (dãy sinh dần), nên bọc bằng `list(...)` để có list.

In [15]:
from itertools import accumulate

it = accumulate([3, 1, 2, 10, 1])
print(it)            # một đối tượng iterator, chưa phải list
print(list(it))      # [3, 4, 6, 16, 17]
# accumulate còn nhận hàm khác phép cộng, ví dụ max cộng dồn:
print(list(accumulate([3, 1, 5, 2], max)))   # [3, 3, 5, 5]

[3, 4, 6, 16, 17]
[3, 3, 5, 5]


## 10. Cách 5 — Prefix sum có số 0 ở đầu (dạng "chuẩn" để tính tổng đoạn)

Nhiều bài cần mảng `prefix` dài `n + 1` với `prefix[0] = 0` và `prefix[k] = nums[0] + ... + nums[k-1]` (tổng `k` phần tử đầu).

Lợi ích: **tổng đoạn** `nums[l] + ... + nums[r] = prefix[r+1] - prefix[l]`, tính trong `O(1)`.

Running sum của bài này chính là `prefix[1:]`.

In [16]:
def build_prefix(nums):
    prefix = [0] * (len(nums) + 1)
    for i, x in enumerate(nums):
        prefix[i + 1] = prefix[i] + x
    return prefix

nums = [3, 1, 2, 10, 1]
prefix = build_prefix(nums)
print("prefix      =", prefix)
print("prefix[1:]  =", prefix[1:], "<- chính là running sum")
l, r = 1, 3
print(f"tổng nums[{l}..{r}] = prefix[{r+1}] - prefix[{l}] = {prefix[r+1]} - {prefix[l]} = {prefix[r+1] - prefix[l]}")
print("kiểm tra trực tiếp:", sum(nums[l:r + 1]))

prefix      = [0, 3, 4, 6, 16, 17]
prefix[1:]  = [3, 4, 6, 16, 17] <- chính là running sum
tổng nums[1..3] = prefix[4] - prefix[1] = 16 - 3 = 13
kiểm tra trực tiếp: 13


`[0] * (n + 1)` tạo list gồm `n + 1` số 0 — cách nhanh để chuẩn bị mảng có độ dài cố định rồi điền dần.

## 11. Vì sao cách 2 đúng?

**Invariant:** ngay sau khi xử lý phần tử thứ `i`, `total = nums[0] + ... + nums[i]`.

- **Khởi đầu:** trước vòng lặp `total = 0` = tổng của 0 phần tử (tổng rỗng).
- **Duy trì:** nếu trước bước `i` có `total = nums[0] + ... + nums[i-1]`, thì sau `total += nums[i]` ta có `total = nums[0] + ... + nums[i]`.
- **Kết quả:** ta `append(total)` ngay sau khi cập nhật, nên `result[i]` đúng bằng định nghĩa `runningSum[i]`. Có đúng `n` lần append → độ dài khớp.

Đây là quy nạp toán học (induction) dạng đơn giản nhất.

## 12. Code nộp LeetCode — lời giải chính

In [17]:
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        result = []
        total = 0
        for x in nums:
            total += x
            result.append(total)
        return result

sol = Solution()
assert sol.runningSum([1, 2, 3, 4]) == [1, 3, 6, 10]
assert sol.runningSum([1, 1, 1, 1, 1]) == [1, 2, 3, 4, 5]
assert sol.runningSum([3, 1, 2, 10, 1]) == [3, 4, 6, 16, 17]
print("PASS 3 ví dụ đề")

PASS 3 ví dụ đề


Các phiên bản nộp khác (đều Accepted):

```python
# In-place
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        for i in range(1, len(nums)):
            nums[i] += nums[i - 1]
        return nums

# Thư viện chuẩn
from itertools import accumulate
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        return list(accumulate(nums))
```

## 13. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ (không tính output) | Ghi chú |
|---|---|---|---|
| 1. `sum(nums[:i+1])` | `O(n²)` | `O(n)` cho slice tạm | dịch thẳng định nghĩa |
| 2. `total` + list mới | `O(n)` | `O(1)` | rõ ràng, không sửa input |
| 3. In-place | `O(n)` | `O(1)` | sửa input |
| 4. `accumulate` | `O(n)` | `O(1)` | ngắn nhất |
| 5. Prefix có số 0 | `O(n)` | `O(n)` mảng prefix | dùng cho truy vấn tổng đoạn |

Ta không thể nhanh hơn `O(n)` vì phải **ghi ra** `n` giá trị kết quả.

Đo thời gian thực tế để thấy khác biệt:

In [18]:
import time, random
random.seed(0)
big = [random.randint(-10**6, 10**6) for _ in range(3000)]
candidates = [("cách 1 sum(slice)", running_sum_bruteforce),
              ("cách 2 total", running_sum_total),
              ("cách 4 accumulate", lambda a: list(accumulate(a)))]
for name, f in candidates:
    t = time.perf_counter()
    f(big)
    print(f"{name:<24}: {(time.perf_counter() - t) * 1000:7.2f} ms")

cách 1 sum(slice)       :   33.14 ms
cách 2 total            :    0.09 ms
cách 4 accumulate       :    0.09 ms


(Số ms cụ thể phụ thuộc máy; điều cần thấy là cách 1 chậm hơn hẳn và chênh lệch tăng nhanh khi `n` lớn.)

## 14. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<22} -> {actual}")

tests = [
    ("đề 1",            [1, 2, 3, 4],       [1, 3, 6, 10]),
    ("đề 2",            [1, 1, 1, 1, 1],    [1, 2, 3, 4, 5]),
    ("đề 3",            [3, 1, 2, 10, 1],   [3, 4, 6, 16, 17]),
    ("1 phần tử",       [7],                [7]),
    ("1 phần tử âm",    [-5],               [-5]),
    ("có số âm",        [5, -2, -4, 3],     [5, 3, -1, 2]),
    ("toàn số 0",       [0, 0, 0],          [0, 0, 0]),
    ("triệt tiêu",      [10**6, -10**6] * 2, [10**6, 0, 10**6, 0]),
]
for name, arr, exp in tests:
    check(name, Solution().runningSum(arr[:]), exp)

PASS  đề 1                   -> [1, 3, 6, 10]
PASS  đề 2                   -> [1, 2, 3, 4, 5]
PASS  đề 3                   -> [3, 4, 6, 16, 17]
PASS  1 phần tử              -> [7]
PASS  1 phần tử âm           -> [-5]
PASS  có số âm               -> [5, 3, -1, 2]
PASS  toàn số 0              -> [0, 0, 0]
PASS  triệt tiêu             -> [1000000, 0, 1000000, 0]


In [20]:
# Đối chiếu 5 cách trên mảng ngẫu nhiên (lưu ý: in-place cần bản sao)
random.seed(2026)
for _ in range(500):
    arr = [random.randint(-10**6, 10**6) for _ in range(random.randint(1, 60))]
    exp = running_sum_bruteforce(arr)
    assert running_sum_total(arr) == exp
    assert running_sum_inplace(arr[:]) == exp
    assert list(accumulate(arr)) == exp
    assert build_prefix(arr)[1:] == exp
    assert len(exp) == len(arr) and exp[-1] == sum(arr)
print("500 mảng ngẫu nhiên: 5 cách khớp nhau; phần tử cuối = sum(nums). PASS")

500 mảng ngẫu nhiên: 5 cách khớp nhau; phần tử cuối = sum(nums). PASS


## 15. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo `total = nums[0]` rồi vẫn duyệt từ đầu → cộng phần tử đầu **hai lần**
```python
total = nums[0]
for x in nums:        # SAI: x đầu tiên lại là nums[0]
    total += x
```

### Lỗi 2: Append **trước** khi cộng → lệch một vị trí
```python
for x in nums:
    result.append(total)   # SAI: lưu tổng của các phần tử TRƯỚC x
    total += x
```
Kết quả là `[0, 1, 3, 6]` — đây là "exclusive prefix sum", không phải đề yêu cầu.

### Lỗi 3: In-place nhưng bắt đầu từ `i = 0`
```python
for i in range(len(nums)):
    nums[i] += nums[i - 1]   # SAI: i=0 thì nums[-1] là PHẦN TỬ CUỐI!
```
Trong Python, chỉ số `-1` không báo lỗi mà lấy phần tử cuối → sai âm thầm.

### Lỗi 4: Dùng `sum(nums[:i])` (thiếu `+1`) → bỏ mất phần tử hiện tại.

In [21]:
def wrong_double_first(nums):
    total, result = nums[0], []
    for x in nums:
        total += x
        result.append(total)
    return result

def wrong_append_before(nums):
    total, result = 0, []
    for x in nums:
        result.append(total)
        total += x
    return result

def wrong_inplace_from_zero(nums):
    for i in range(len(nums)):
        nums[i] += nums[i - 1]
    return nums

def wrong_slice(nums):
    return [sum(nums[:i]) for i in range(len(nums))]

a = [1, 2, 3, 4]
print("đúng                 :", running_sum_total(a))
print("lỗi 1 cộng đầu 2 lần :", wrong_double_first(a))
print("lỗi 2 append trước   :", wrong_append_before(a))
print("lỗi 3 in-place từ 0  :", wrong_inplace_from_zero(a[:]))
print("lỗi 4 thiếu +1       :", wrong_slice(a))

đúng                 : [1, 3, 6, 10]
lỗi 1 cộng đầu 2 lần : [2, 4, 7, 11]
lỗi 2 append trước   : [0, 1, 3, 6]
lỗi 3 in-place từ 0  : [5, 7, 10, 14]
lỗi 4 thiếu +1       : [0, 1, 3, 6]


Lỗi 3 phân tích: `i=0` → `nums[0] += nums[-1]` = `1 + 4 = 5`; từ đó mọi giá trị sau đều bị cộng dư 4.

### Mẹo nhận dạng pattern

Nghĩ ngay tới **prefix sum / running sum** khi đề có:
- "tổng từ đầu tới vị trí i", "số dư", "tổng tích lũy", "điểm cộng dồn"…
- nhiều câu hỏi "tổng đoạn `[l, r]`" trên cùng một mảng (LeetCode 303 Range Sum Query).
- "chỉ số cân bằng: tổng trái = tổng phải" (LeetCode 724 Find Pivot Index).
- "đếm số đoạn con có tổng bằng k" (LeetCode 560, prefix sum + dict).
- "giá trị khởi đầu nhỏ nhất để tổng từng bước luôn dương" (LeetCode 1413).

Câu thần chú: **"giá trị hôm nay = giá trị hôm qua + thay đổi hôm nay"**.

### Biến thể

### Biến thể A — Tích cộng dồn (running product)

In [22]:
import operator
print(list(accumulate([1, 2, 3, 4], operator.mul)))   # [1, 2, 6, 24]

[1, 2, 6, 24]


### Biến thể B — Pivot index (LeetCode 724)

Tìm chỉ số `i` nhỏ nhất mà tổng bên trái = tổng bên phải (không tính `nums[i]`). Dùng `left` chạy và `total`:
`right = total - left - nums[i]`.

In [23]:
def pivot_index(nums):
    total, left = sum(nums), 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

print(pivot_index([1, 7, 3, 6, 5, 6]))   # 3
print(pivot_index([1, 2, 3]))            # -1

3
-1


## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (đoán output)
Không chạy, đoán `running_sum_total([2, -2, 2, -2])`. Viết hàm trả về list bạn đoán.

In [24]:
def exercise_predict() -> list[int]:
    # TODO: trả về list bạn đoán
    pass

# assert exercise_predict() == running_sum_total([2, -2, 2, -2])

### Bài 2 (điền chỗ trống — in-place)

In [25]:
def exercise_running_sum_inplace(nums: list[int]) -> list[int]:
    # TODO: vòng for bắt đầu từ chỉ số nào?
    # TODO: nums[i] = ?
    pass

# assert exercise_running_sum_inplace([1, 2, 3, 4]) == [1, 3, 6, 10]
# assert exercise_running_sum_inplace([5]) == [5]

### Bài 3 (tổng đoạn bằng prefix)
Viết `range_sum(nums, queries)`: với mỗi truy vấn `(l, r)` trả về `nums[l] + ... + nums[r]`. Yêu cầu: dựng prefix **một lần**, mỗi truy vấn `O(1)`.

In [26]:
def exercise_range_sum(nums: list[int], queries: list[tuple[int, int]]) -> list[int]:
    # TODO
    pass

# assert exercise_range_sum([-2, 0, 3, -5, 2, -1], [(0, 2), (2, 5), (0, 5)]) == [1, -1, -3]

### Bài 4 (LeetCode 1413 — giá trị khởi đầu nhỏ nhất)
Cho `nums`, tìm số nguyên dương `start` nhỏ nhất sao cho mọi tổng cộng dồn `start + nums[0] + ... + nums[i]` đều `>= 1`.

Ví dụ `[-3, 2, -3, 4, 2]` → `5`.

In [27]:
def exercise_min_start_value(nums: list[int]) -> int:
    # TODO: tìm running sum nhỏ nhất
    pass

# assert exercise_min_start_value([-3, 2, -3, 4, 2]) == 5
# assert exercise_min_start_value([1, 2]) == 1

### Bài 5 (running sum ngược — suffix sum)
Trả về `suffix[i] = nums[i] + nums[i+1] + ... + nums[n-1]`.

Ví dụ `[1, 2, 3, 4]` → `[10, 9, 7, 4]`.

In [28]:
def exercise_suffix_sum(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_suffix_sum([1, 2, 3, 4]) == [10, 9, 7, 4]
# assert exercise_suffix_sum([5]) == [5]

### Bài 6 (mini challenge — LeetCode 560)
Đếm số **đoạn con liên tiếp** có tổng đúng bằng `k`. Yêu cầu `O(n)`.

Ví dụ `nums = [1, 1, 1], k = 2` → `2`; `nums = [1, 2, 3], k = 3` → `2`.

In [29]:
def exercise_subarray_sum_equals_k(nums: list[int], k: int) -> int:
    # TODO: prefix sum + dict đếm số lần mỗi prefix đã xuất hiện
    pass

# assert exercise_subarray_sum_equals_k([1, 1, 1], 2) == 2
# assert exercise_subarray_sum_equals_k([1, 2, 3], 3) == 2
# assert exercise_subarray_sum_equals_k([1, -1, 0], 0) == 3

### Gợi ý ba mức

**Bài 1** — M1: mỗi bước cộng thêm phần tử. M2: lập bảng như mục 7. M3: tổng dao động giữa 2 và 0.

**Bài 2** — M1: vị trí 0 giữ nguyên. M2: `range(1, len(nums))`. M3: `nums[i] += nums[i - 1]` rồi `return nums`.

**Bài 3** — M1: dựng `prefix` dài `n+1`. M2: tổng đoạn `[l, r]` = `prefix[r+1] - prefix[l]`. M3: `return [prefix[r + 1] - prefix[l] for l, r in queries]`.

**Bài 4** — M1: chỉ running sum **nhỏ nhất** là quan trọng. M2: cần `start + min_prefix >= 1`. M3: `return max(1, 1 - min_prefix)` với `min_prefix` là min của running sum.

**Bài 5** — M1: giống running sum nhưng đi từ phải sang trái. M2: `range(n - 1, -1, -1)`. M3: tạo `res = [0]*n`, `total += nums[i]; res[i] = total`.

**Bài 6** — M1: tổng đoạn `(j, i]` = `prefix_i - prefix_j`; muốn bằng `k` thì cần `prefix_j = prefix_i - k`. M2: dict `count` lưu số lần mỗi prefix đã thấy, khởi tạo `{0: 1}`. M3: mỗi bước `ans += count.get(total - k, 0)` rồi `count[total] = count.get(total, 0) + 1`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [30]:
def answer_predict():
    return [2, 0, 2, 0]

def answer_running_sum_inplace(nums):
    for i in range(1, len(nums)):
        nums[i] += nums[i - 1]
    return nums

def answer_range_sum(nums, queries):
    prefix = [0] * (len(nums) + 1)
    for i, x in enumerate(nums):
        prefix[i + 1] = prefix[i] + x
    return [prefix[r + 1] - prefix[l] for l, r in queries]

def answer_min_start_value(nums):
    total, min_prefix = 0, 0
    for x in nums:
        total += x
        min_prefix = min(min_prefix, total)
    return max(1, 1 - min_prefix)

def answer_suffix_sum(nums):
    n = len(nums)
    res, total = [0] * n, 0
    for i in range(n - 1, -1, -1):
        total += nums[i]
        res[i] = total
    return res

def answer_subarray_sum_equals_k(nums, k):
    count = {0: 1}        # prefix rỗng = 0 xuất hiện 1 lần
    total = ans = 0
    for x in nums:
        total += x
        ans += count.get(total - k, 0)
        count[total] = count.get(total, 0) + 1
    return ans

assert answer_predict() == running_sum_total([2, -2, 2, -2])
assert answer_running_sum_inplace([1, 2, 3, 4]) == [1, 3, 6, 10]
assert answer_running_sum_inplace([5]) == [5]
assert answer_range_sum([-2, 0, 3, -5, 2, -1], [(0, 2), (2, 5), (0, 5)]) == [1, -1, -3]
assert answer_min_start_value([-3, 2, -3, 4, 2]) == 5
assert answer_min_start_value([1, 2]) == 1
assert answer_suffix_sum([1, 2, 3, 4]) == [10, 9, 7, 4]
assert answer_suffix_sum([5]) == [5]
assert answer_subarray_sum_equals_k([1, 1, 1], 2) == 2
assert answer_subarray_sum_equals_k([1, 2, 3], 3) == 2
assert answer_subarray_sum_equals_k([1, -1, 0], 0) == 3

# Đối chiếu bài 6 với brute force O(n^2)
def brute_k(nums, k):
    return sum(1 for i in range(len(nums)) for j in range(i, len(nums)) if sum(nums[i:j + 1]) == k)
random.seed(5)
for _ in range(300):
    arr = [random.randint(-3, 3) for _ in range(random.randint(1, 12))]
    k = random.randint(-4, 4)
    assert answer_subarray_sum_equals_k(arr, k) == brute_k(arr, k), (arr, k)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def runningSum(self, nums: list[int]) -> list[int]:
        result, total = [], 0
        for x in nums:
            total += x            # hôm nay = hôm qua + thay đổi hôm nay
            result.append(total)
        return result
```

Template prefix sum cho truy vấn tổng đoạn:
```python
prefix = [0] * (n + 1)
for i, x in enumerate(nums):
    prefix[i + 1] = prefix[i] + x
# tổng nums[l..r] = prefix[r + 1] - prefix[l]
```

**Checklist trước khi submit:**
- [ ] `total` bắt đầu từ `0` (hoặc in-place bắt đầu từ `i = 1`)?
- [ ] Cộng **trước**, append **sau**?
- [ ] Output dài đúng `len(nums)`?
- [ ] Có xét số âm / mảng 1 phần tử?
- [ ] Nếu sửa in-place: có chấp nhận việc input bị đổi không?